# 0 — Download data from the BioImage Archive

Fetches the CellProfiler feature tables from **S-BIAD2254** and lays them out the way
`1_FeatureSorting.ipynb` expects. 18 files, **16.6 GB**; re-running skips whatever is
already on disk.

You do not need this to draw the figures. The figures run from the processed profile
tables in `data/`, which `scripts/download_data.py` fetches in ~550 MB. This notebook is
for re-deriving those tables from the CellProfiler output.

The plate metadata is **not** downloaded: `spher_colo52-metadata.csv` and `filemap.csv`
ship with the repo, and the shipped copies carry columns the deposit's FileList does not
(`layout_id`, `plate_well`, `cmpd_conc`, `smiles`, …) which the figures depend on.

Raw OME-TIFFs are not downloaded either — see the last cell if you want them.


In [ ]:
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import DATA_ROOT, metadata, require

import pandas as pd

sys.path.insert(0, str(ROOT / "scripts"))
from download_data import BIA_FILES_URL, check_parquet_intact, fetch_url

PLATES = ["PB000137", "PB000138", "PB000139", "PB000140", "PB000141", "PB000142"]
OBJECTS = ["featICF_nuclei", "featICF_cells", "featICF_cytoplasm"]

# Lands under data/, which is gitignored, and is where cellprofiler_results() looks.
sourceDir = DATA_ROOT / "cellprofiler_results" / "exp1_main"
sourceDir.mkdir(parents=True, exist_ok=True)
print("downloading into", sourceDir)

## Where each plate goes

`1_FeatureSorting` builds its per-plate path as `{sourceDir}/{barcode}/{image_id}/{cp_id}`,
taking `image_id` and `cp_id` from the shipped metadata. Read them from there rather than
inventing them, so the download lands where the next notebook will look.


In [ ]:
meta = pd.read_csv(require(metadata("spher_colo52-metadata.csv", "exp1_main")))
ids = (meta[["barcode", "image_id", "cp_id"]]
       .drop_duplicates()
       .set_index("barcode"))

missing = [bc for bc in PLATES if bc not in ids.index]
assert not missing, f"no image_id/cp_id in the shipped metadata for {missing}"

for bc in PLATES:
    print(f"  {bc} -> {ids.loc[bc, 'image_id']}/{ids.loc[bc, 'cp_id']}")

## Feature tables

16.6 GB in 18 files. Already-downloaded files are skipped, so this is safe to re-run.


In [ ]:
total = 0
bad = []
for bc in PLATES:
    dest_dir = sourceDir / bc / str(ids.loc[bc, "image_id"]) / str(ids.loc[bc, "cp_id"])
    for obj in OBJECTS:
        out = dest_dir / f"{obj}.parquet"
        had = out.exists() and out.stat().st_size > 0
        fetch_url(f"{BIA_FILES_URL}/results/{bc}/{obj}.parquet", out)
        total += out.stat().st_size

        # Verify the file is a complete parquet, not just the right number of bytes.
        # A truncated upload still serves a matching Content-Length, so size proves
        # nothing; the closing PAR1 footer does.
        complaint = check_parquet_intact(out)
        if complaint:
            bad.append((f"{bc}/{obj}", complaint))
        print(f"  {'skipped ' if had else 'fetched '} {out.relative_to(sourceDir)}"
              f"  {out.stat().st_size / 1e6:6.0f} MB  {complaint or 'ok'}")

print(f"\n{total / 1e9:.2f} GB present")
if bad:
    print(f"\n{len(bad)} of {len(PLATES) * len(OBJECTS)} files are unusable:")
    for name, why in bad:
        print(f"    {name}: {why}")
    raise SystemExit(
        "The deposited CellProfiler tables are incomplete — this is a problem with the "
        "archived files, not with the download. Re-upload them before re-running; the "
        "figures do not need them (see scripts/download_data.py)."
    )

## Check

`1_FeatureSorting` picks this up on its own — `cellprofiler_results("exp1_main")` prefers
this download over the cluster mount. Nothing to edit by hand.


In [ ]:
from utils.paths import cellprofiler_results

for bc in PLATES:
    d = sourceDir / bc / str(ids.loc[bc, "image_id"]) / str(ids.loc[bc, "cp_id"])
    for obj in OBJECTS:
        assert (d / f"{obj}.parquet").stat().st_size > 0, f"empty or missing: {d/obj}"

resolved = cellprofiler_results("exp1_main")
print("all 18 parquets present")
print("1_FeatureSorting will read from:", resolved)
assert resolved == sourceDir, (
    f"resolver points at {resolved}, not the download. Unset COLOPAINT3D_CP_RESULTS.")

---

Raw images (129,281 OME-TIFFs) are not needed for the feature pipeline:

```bash
wget -r -np -nH --cut-dirs=5 -c \
  https://ftp.ebi.ac.uk/biostudies/fire/S-BIAD/254/S-BIAD2254/Files/spher-colo52/
```
